# 02 — SciPy — interpolation

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- interpoler des données 1D avec `interp1d`, `CubicSpline`, `PchipInterpolator` ;
- comprendre la différence entre interpolation et fitting ;
- interpoler des données 2D avec `griddata` et `RegularGridInterpolator` ;
- lisser des données bruitées avec `UnivariateSpline` ;
- calculer les dérivées et intégrales d’une spline ;
- gérer l’extrapolation et ses pièges.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- NumPy (arrays, linspace, meshgrid) ;
- bases de matplotlib (pour comprendre les exemples).

Ce que ce notebook **ne couvrira pas** :

- le machine learning et la régression ;
- la géostatistique (kriging).

## Plan

1. Installation
2. `interp1d` — interpolation 1D basique
3. Modes d’interpolation
4. `CubicSpline`
5. `PchipInterpolator`
6. Comparaison des méthodes 1D
7. `griddata` — interpolation 2D
8. `RegularGridInterpolator`
9. `UnivariateSpline` — lissage
10. Dérivées et intégrales
11. Extrapolation
12. Synthèse
13. Exercices
14. Ressources externes

### Exemple : interpolation de données météo

Interpolons des relevés de température horaire pour estimer
la température toutes les 15 minutes.

In [ ]:
import numpy as np
from scipy.interpolate import interp1d, CubicSpline
import matplotlib.pyplot as plt

# Température relevée toutes les 3h
heures = np.array([0, 3, 6, 9, 12, 15, 18, 21, 24])
temps = np.array([12, 10, 9, 14, 22, 24, 20, 15, 12])

# Interpolation
h_fine = np.linspace(0, 24, 97)  # toutes les 15 min
f_linear = interp1d(heures, temps, kind='linear')
f_cubic = CubicSpline(heures, temps)

plt.figure(figsize=(10, 4))
plt.plot(heures, temps, 'ko', ms=8, label='relevés')
plt.plot(h_fine, f_linear(h_fine), '--', label='linéaire')
plt.plot(h_fine, f_cubic(h_fine), '-', label='cubic spline')
plt.xlabel('Heure'); plt.ylabel('Température (°C)')
plt.legend(); plt.grid(True); plt.show()

### Exercice : Runge phenomenon

Le phénomène de Runge montre que l'interpolation polynomiale de haut degré
peut osciller fortement aux bords de l'intervalle.

1. Interpolez $f(x) = \frac{1}{1 + 25x^2}$ sur $[-1, 1]$ avec 11, 15 et 21 points équidistants
2. Comparez avec une interpolation par splines cubiques
3. Observez le comportement aux bords

In [ ]:
from scipy.interpolate import BarycentricInterpolator, CubicSpline

f_runge = lambda x: 1 / (1 + 25*x**2)
x_fine = np.linspace(-1, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, [11, 15, 21]):
    x_pts = np.linspace(-1, 1, n)
    y_pts = f_runge(x_pts)
    poly = BarycentricInterpolator(x_pts, y_pts)
    cs = CubicSpline(x_pts, y_pts)

    ax.plot(x_fine, f_runge(x_fine), 'k-', lw=2, label='f(x)')
    ax.plot(x_fine, poly(x_fine), 'r--', label=f'poly n={n}')
    ax.plot(x_fine, cs(x_fine), 'b-', alpha=0.7, label='spline')
    ax.plot(x_pts, y_pts, 'ko', ms=4)
    ax.set_ylim(-1, 2); ax.legend(fontsize=8)
    ax.set_title(f'{n} points')
plt.tight_layout(); plt.show()

### Exemple avancé : interpolation 2D pour une carte de terrain

Utilisons `griddata` pour créer une surface lisse à partir de mesures d'altitude dispersées.

In [ ]:
from scipy.interpolate import griddata

# Points d'altitude mesurés aléatoirement
np.random.seed(0)
n_pts = 100
x = np.random.rand(n_pts) * 10
y = np.random.rand(n_pts) * 10
z = np.sin(x) * np.cos(y/2) + 0.3 * np.random.randn(n_pts)

# Grille régulière
xi = np.linspace(0, 10, 100)
yi = np.linspace(0, 10, 100)
Xi, Yi = np.meshgrid(xi, yi)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, method in zip(axes, ['nearest', 'linear', 'cubic']):
    Zi = griddata((x, y), z, (Xi, Yi), method=method)
    im = ax.pcolormesh(Xi, Yi, Zi, shading='auto', cmap='terrain')
    ax.scatter(x, y, c='red', s=5)
    ax.set_title(method); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

### Exercice récapitulatif

1. Générez un signal sinusoidal échantillonné à 10 Hz sur 2 secondes
2. Sous-échantillonnez à 3 Hz (gardez 1 point sur ~3)
3. Reconstruisez le signal original par interpolation cubique
4. Calculez l'erreur RMS entre le signal original et reconstruit
5. Tracez les trois signaux (original, sous-échantillonné, reconstruit)

In [ ]:
# Signal original
fs = 10  # Hz
t = np.arange(0, 2, 1/fs)
signal = np.sin(2*np.pi*1.2*t) + 0.5*np.cos(2*np.pi*0.5*t)

# Sous-échantillonnage
t_sub = t[::3]
s_sub = signal[::3]

# Reconstruction
cs = CubicSpline(t_sub, s_sub)
s_recon = cs(t)

rms = np.sqrt(np.mean((signal - s_recon)**2))
print(f'Erreur RMS : {rms:.4f}')

plt.figure(figsize=(10, 4))
plt.plot(t, signal, 'k-', label='original')
plt.plot(t_sub, s_sub, 'ro', ms=6, label='sous-échantillonné')
plt.plot(t, s_recon, 'b--', label='reconstruit')
plt.legend(); plt.grid(); plt.show()

---

## 1. Installation

In [ ]:
import numpy as np
import scipy
print(f"SciPy {scipy.__version__}")

---

## 2. `interp1d` — interpolation 1D basique

Crée une fonction appelable qui passe par les points donnés.

In [ ]:
from scipy.interpolate import interp1d

x = np.linspace(0, 10, 11)  # 11 points
y = np.sin(x)
f_lin = interp1d(x, y, kind='linear')
f_lin(2.5)

In [ ]:
f_cub = interp1d(x, y, kind='cubic')
f_cub(2.5)

In [ ]:
# Comparer avec la vraie valeur
print(f"Vrai:     {np.sin(2.5):.8f}")
print(f"Linéaire: {f_lin(2.5):.8f}")
print(f"Cubique:  {f_cub(2.5):.8f}")

---

## 3. Modes d’interpolation

| `kind` | Description | Continuité |
|---|---|---|
| `'linear'` | Linéaire par morceaux | C0 |
| `'quadratic'` | Spline degré 2 | C1 |
| `'cubic'` | Spline degré 3 | C2 |
| `'nearest'` | Valeur du point le plus proche | C−1 |
| `'zero'` | Valeur constante par morceaux | C−1 |
| `'previous'` / `'next'` | Valeur précédente / suivante | C−1 |

In [ ]:
for kind in ['nearest', 'zero', 'linear', 'quadratic', 'cubic']:
    f = interp1d(x, y, kind=kind)
    print(f"{kind:12s} f(2.5) = {f(2.5):.6f}")

---

## 4. `CubicSpline`

Plus moderne et flexible que `interp1d(kind='cubic')`.

In [ ]:
from scipy.interpolate import CubicSpline

cs = CubicSpline(x, y)
cs(2.5)

In [ ]:
# Dérivée première
cs(2.5, 1)

In [ ]:
# Dérivée seconde
cs(2.5, 2)

### Conditions aux limites

Par défaut : *not-a-knot*. On peut spécifier les dérivées aux bords.

In [ ]:
# Conditions naturelles (dérivée seconde = 0 aux bords)
cs_nat = CubicSpline(x, y, bc_type='natural')
print(f"Nat: {cs_nat(2.5):.8f}")

# Clamped (dérivée première imposée)
cs_clamp = CubicSpline(x, y, bc_type='clamped')
print(f"Clamp: {cs_clamp(2.5):.8f}")

---

## 5. `PchipInterpolator`

PCHIP préserve la monotonicité : pas d’oscillations parasites entre les points.

In [ ]:
from scipy.interpolate import PchipInterpolator

pch = PchipInterpolator(x, y)
pch(2.5)

Utile quand les données sont monotones ou présentent des discontinuités de pente.

---

## 6. Comparaison des méthodes 1D

In [ ]:
xs = np.linspace(0, 10, 100)
print(f"{'Méthode':15s} {'Erreur max':>12s}")
for name, f in [('interp1d lin', interp1d(x, y)), ('interp1d cub', interp1d(x, y, kind='cubic')),
                ('CubicSpline', CubicSpline(x, y)), ('PCHIP', PchipInterpolator(x, y))]:
    err = np.max(np.abs(f(xs) - np.sin(xs)))
    print(f"{name:15s} {err:12.2e}")

---

## 7. `griddata` — interpolation 2D

Pour des points dispersés (non structurés).

In [ ]:
from scipy.interpolate import griddata

points = np.array([[0, 0], [1, 0], [0, 1], [1, 1], [0.5, 0.5]])
values = np.array([0, 1, 1, 2, 1.5])

grid_x, grid_y = np.mgrid[0:1:20j, 0:1:20j]
grid_z = griddata(points, values, (grid_x, grid_y), method='cubic')
print(f"Grille shape: {grid_z.shape}")

In [ ]:
# Méthodes disponibles
for method in ['nearest', 'linear', 'cubic']:
    gz = griddata(points, values, (grid_x, grid_y), method=method)
    print(f"{method:8s}: NaN count = {np.isnan(gz).sum()}")

---

## 8. `RegularGridInterpolator`

Pour des données sur une grille régulière (plus rapide que `griddata`).

In [ ]:
from scipy.interpolate import RegularGridInterpolator

xg = np.linspace(0, 5, 6)
yg = np.linspace(0, 5, 6)
zg = np.sin(xg[:, None]) * np.cos(yg[None, :])

rgi = RegularGridInterpolator((xg, yg), zg)
rgi([[2.3, 3.7]])

In [ ]:
# Plusieurs points d'un coup
pts = np.array([[1.1, 2.2], [3.3, 4.4], [0.5, 0.5]])
rgi(pts)

---

## 9. `UnivariateSpline` — lissage

Contrairement à l’interpolation, le lissage ne passe pas forcément par tous les points.

In [ ]:
from scipy.interpolate import UnivariateSpline

rng = np.random.default_rng(42)
xn = np.linspace(0, 10, 30)
yn = np.sin(xn) + 0.3 * rng.standard_normal(30)

spl = UnivariateSpline(xn, yn, s=1.0)
print(f"spl(5.5) = {spl(5.5):.4f}")

Le paramètre `s` contrôle le lissage :
- `s=0` : interpolation exacte (passe par tous les points)
- `s` grand : lissage fort (moins de knots)

In [ ]:
for s in [0, 0.5, 1.0, 5.0, 20.0]:
    spl = UnivariateSpline(xn, yn, s=s)
    print(f"s={s:5.1f}: knots={len(spl.get_knots()):2d}, spl(5)={spl(5):.4f}")

---

## 10. Dérivées et intégrales

In [ ]:
cs = CubicSpline(x, y)

# Dérivée
print(f"f'(2.5) = {cs(2.5, 1):.6f} (vrai cos(2.5) = {np.cos(2.5):.6f})")

In [ ]:
# Intégrale définie
from scipy.integrate import quad
integral_spline = cs.integrate(0, np.pi)
integral_exact, _ = quad(np.sin, 0, np.pi)
print(f"Intégrale spline: {integral_spline:.6f}")
print(f"Intégrale exacte:  {integral_exact:.6f}")

In [ ]:
# Racines de la spline
print(f"Racines: {cs.roots()}")

---

## 11. Extrapolation

Par défaut, `interp1d` refuse d’évaluer hors du domaine.

In [ ]:
f = interp1d(x, y)
try:
    f(15)
except ValueError as e:
    print(f"Erreur: {e}")

In [ ]:
# Activer l'extrapolation
f_ext = interp1d(x, y, fill_value='extrapolate')
print(f"f(15) = {f_ext(15):.4f}  (attention, extrapolation !)")

**Attention** : l’extrapolation est souvent dangereuse. Les splines divergent vite hors du domaine.

In [ ]:
# CubicSpline extrapole par défaut
cs = CubicSpline(x, y)
print(f"cs(15) = {cs(15):.4f}  (vrai sin(15) = {np.sin(15):.4f})")

---

## Synthèse

| Usage | Fonction | Points clés |
|---|---|---|
| 1D basique | `interp1d(x, y, kind=)` | Simple mais déprécié |
| Spline C2 | `CubicSpline(x, y)` | Dérivées, intégrales, racines |
| Monotone | `PchipInterpolator(x, y)` | Pas d’oscillations |
| 2D dispersé | `griddata(pts, vals, grid)` | Points non structurés |
| Grille régulière | `RegularGridInterpolator` | Plus rapide |
| Lissage | `UnivariateSpline(x, y, s=)` | `s` contrôle le compromis |

### Règles à retenir

- **Interpolation** passe par tous les points ; **lissage** non.
- **CubicSpline** est C2 (dérivée seconde continue).
- **PCHIP** est idéal pour les données monotones.
- **`fill_value='extrapolate'`** pour sortir du domaine (avec prudence).
- **`s=0`** dans `UnivariateSpline` = interpolation exacte.

---

---

## Interpolation B-splines

In [ ]:
from scipy.interpolate import make_interp_spline
import numpy as np
x = np.linspace(0, 10, 15)
y = np.sin(x)
spl = make_interp_spline(x, y, k=3)
xs = np.linspace(0, 10, 100)
err = np.max(np.abs(spl(xs) - np.sin(xs)))
print(f"B-spline k=3: erreur max = {err:.2e}")

In [ ]:
for k in [1, 2, 3, 5]:
    spl = make_interp_spline(x, y, k=k)
    err = np.max(np.abs(spl(xs) - np.sin(xs)))
    print(f"k={k}: erreur max = {err:.2e}")

### Interpolation avec poids

In [ ]:
from scipy.interpolate import UnivariateSpline
import numpy as np
rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = np.sin(x) + 0.2 * rng.standard_normal(30)
w = np.ones(30)
w[15] = 10.0  # point 15 a plus de poids
spl = UnivariateSpline(x, y, w=w, s=1)
print(f"spl(5) = {spl(5):.4f}")

---

## Interpolation Akima

In [ ]:
from scipy.interpolate import Akima1DInterpolator
import numpy as np
x = np.linspace(0, 10, 11)
y = np.sin(x)
ak = Akima1DInterpolator(x, y)
print(f"Akima(2.5) = {ak(2.5):.8f}")

Akima est un compromis entre CubicSpline et PCHIP : elle evite les oscillations tout en restant lisse.

## Exercices

### Exercice 1 *(facile)* — Interpolation linéaire

Interpolez linéairement `sin(x)` échantillonnée en 5 points sur `[0, pi]`, et évaluez en 10 points.

In [ ]:
from scipy.interpolate import interp1d
import numpy as np

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.interpolate import interp1d

x = np.linspace(0, np.pi, 5)
y = np.sin(x)
f = interp1d(x, y)
xs = np.linspace(0, np.pi, 10)
print(f(xs))
```

</details>

### Exercice 2 *(facile)* — CubicSpline

Interpolez `x=[0,1,2,3,4,5]`, `y=[0,1,4,9,16,25]` (parabole) avec `CubicSpline`. Comparez `cs(2.5)` au vrai `6.25`.

In [ ]:
from scipy.interpolate import CubicSpline
import numpy as np

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from scipy.interpolate import CubicSpline
import numpy as np

x = np.arange(6)
y = x**2
cs = CubicSpline(x, y)
print(f"cs(2.5) = {cs(2.5):.4f} (attendu 6.25)")
```

</details>

### Exercice 3 *(moyen)* — Dérivée d’une spline

À partir d’une spline de `sin(x)` sur `[0, 2*pi]` en 20 points, calculez `spline'(x)` et comparez avec `cos(x)` en 10 points.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.interpolate import CubicSpline

x = np.linspace(0, 2 * np.pi, 20)
cs = CubicSpline(x, np.sin(x))
xs = np.linspace(0, 2 * np.pi, 10)
for xi in xs:
    print(f"x={xi:.2f}: spline'={cs(xi, 1):.4f}, cos={np.cos(xi):.4f}")
```

</details>

### Exercice 4 *(moyen)* — Interpolation 2D

Générez 30 points aléatoires dans `[0,1]²` avec `z = sin(5x) * cos(5y)`. Interpolez sur une grille 50x50 avec `griddata`.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.interpolate import griddata

rng = np.random.default_rng(0)
pts = rng.random((30, 2))
z = np.sin(5 * pts[:, 0]) * np.cos(5 * pts[:, 1])
gx, gy = np.mgrid[0:1:50j, 0:1:50j]
gz = griddata(pts, z, (gx, gy), method='cubic')
print(f"Shape: {gz.shape}, NaN: {np.isnan(gz).sum()}")
```

</details>

### Exercice 5 *(difficile)* — Spline de lissage optimale

Générez des données bruitées `y = sin(x) + bruit` (50 points, `[0, 10]`). Testez `UnivariateSpline` avec `s` variant de 0.1 à 20. Trouvez le `s` qui minimise l’erreur par rapport à `sin(x)` évaluée sur 200 points.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.interpolate import UnivariateSpline

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 50)
y = np.sin(x) + 0.3 * rng.standard_normal(50)
xs = np.linspace(0, 10, 200)
best_s, best_err = 0, float('inf')
for s in np.logspace(-1, 1.3, 30):
    spl = UnivariateSpline(x, y, s=s)
    err = np.mean((spl(xs) - np.sin(xs))**2)
    if err < best_err:
        best_s, best_err = s, err
print(f"Meilleur s={best_s:.3f}, MSE={best_err:.6f}")
```

</details>

### Approfondir : interpolation avec RegularGridInterpolator

Pour interpoler des données sur une grille régulière en N dimensions,
`RegularGridInterpolator` est plus efficace que `griddata`.

In [ ]:
from scipy.interpolate import RegularGridInterpolator
import numpy as np

# Grille 2D régulière
x = np.linspace(0, 4, 20)
y = np.linspace(0, 4, 20)
data = np.sin(x[:, None]) * np.cos(y[None, :])

interp = RegularGridInterpolator((x, y), data, method='cubic')

# Évaluation en des points quelconques
pts = np.array([[1.5, 2.3], [0.7, 3.1], [3.0, 1.0]])
print('Valeurs interpolées :', interp(pts))
print('Valeurs exactes     :', np.sin(pts[:,0]) * np.cos(pts[:,1]))

### Exercice : interpolation de données manquantes

Un capteur a des trous dans ses mesures. Utilisez l'interpolation pour combler :

1. Créez un signal sinusoidal de 100 points
2. Supprimez aléatoirement 30% des points (mettez-les à NaN)
3. Interpolez les NaN avec `interp1d`
4. Comparez l'erreur pour `linear`, `quadratic` et `cubic`

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

np.random.seed(42)
x = np.linspace(0, 4*np.pi, 100)
y_true = np.sin(x) + 0.3*np.sin(3*x)

# Créer des trous
mask = np.random.random(100) > 0.3
mask[0] = mask[-1] = True  # Garder les bords
x_known = x[mask]
y_known = y_true[mask]

plt.figure(figsize=(12, 4))
plt.plot(x, y_true, 'k--', alpha=0.3, label='vrai signal')
plt.plot(x_known, y_known, 'ko', ms=3, label='données connues')

for kind, color in [('linear','blue'), ('quadratic','orange'), ('cubic','green')]:
    f = interp1d(x_known, y_known, kind=kind)
    y_interp = f(x)
    rmse = np.sqrt(np.mean((y_true - y_interp)**2))
    plt.plot(x, y_interp, color=color, alpha=0.7, label=f'{kind} (RMSE={rmse:.4f})')

plt.legend(); plt.grid(); plt.show()

### Astuce : RBF interpolation pour des données dispersées

`RBFInterpolator` (nouveau dans SciPy 1.7) remplace l'ancien `Rbf`.
Il est parfait pour les données dispersées en N dimensions.

In [ ]:
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt

# Points dispersés
np.random.seed(0)
xy = np.random.rand(50, 2) * 6
z = np.sin(xy[:,0]) + np.cos(xy[:,1])

# Interpolation RBF
rbf = RBFInterpolator(xy, z, kernel='thin_plate_spline')

# Grille d'évaluation
xi = np.linspace(0, 6, 80)
yi = np.linspace(0, 6, 80)
Xi, Yi = np.meshgrid(xi, yi)
pts_eval = np.column_stack([Xi.ravel(), Yi.ravel()])
Zi = rbf(pts_eval).reshape(Xi.shape)

plt.figure(figsize=(8, 6))
plt.pcolormesh(Xi, Yi, Zi, shading='auto', cmap='RdYlBu_r')
plt.scatter(xy[:,0], xy[:,1], c='black', s=20)
plt.colorbar(label='z'); plt.title('RBF interpolation')
plt.show()

### Exercice bonus : interpolation de trajectoire GPS

Un GPS enregistre la position toutes les 5 secondes, mais vous voulez une trajectoire lisse.

1. Simulez 20 points GPS (longitude, latitude) le long d'un chemin sinueux
2. Interpolez avec `CubicSpline` en mode paramétrique (un spline pour x(t), un pour y(t))
3. Tracez la trajectoire brute et interpolée

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt

# Simulation GPS
np.random.seed(7)
n = 20
t = np.arange(n)
lon = 2.35 + 0.01 * t + 0.003 * np.sin(t) + 0.001 * np.random.randn(n)
lat = 48.85 + 0.005 * t + 0.002 * np.cos(t * 1.5) + 0.0005 * np.random.randn(n)

# Interpolation paramétrique
t_fine = np.linspace(0, n-1, 200)
cs_lon = CubicSpline(t, lon)
cs_lat = CubicSpline(t, lat)

plt.figure(figsize=(8, 6))
plt.plot(lon, lat, 'ro-', ms=6, label='GPS brut')
plt.plot(cs_lon(t_fine), cs_lat(t_fine), 'b-', alpha=0.7, label='interpolé')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.legend(); plt.grid(); plt.title('Trajectoire GPS interpolée')
plt.show()

### Astuce : choisir le bon type d'interpolation

| Méthode | Quand l'utiliser |
|---------|------------------|
| `interp1d` linear | Données bruitées, pas besoin de lissage |
| `CubicSpline` | Données propres, besoin de continuité C2 |
| `PchipInterpolator` | Besoin de monotonicité (pas de dépassement) |
| `BarycentricInterpolator` | Peu de points, polynôme exact |
| `RBFInterpolator` | Données dispersées en N dimensions |
| `RegularGridInterpolator` | Grille régulière ND, rapide |
| `griddata` | Données dispersées 2D/3D |

---

## Ressources externes

- [scipy.interpolate — documentation officielle](https://docs.scipy.org/doc/scipy/reference/interpolate.html)
- [Interpolation tutorial](https://docs.scipy.org/doc/scipy/tutorial/interpolate.html)